In [1]:
# Import + path + load

import os
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from sklearn.preprocessing import StandardScaler

DATA_PROCESSED_DIR = "../data/processed"
LOOKBACK = 60
OUT_DIR = os.path.join(DATA_PROCESSED_DIR, f"datasets_close_L{LOOKBACK}")

TICKER = "BTC-USD"
H = 1
npz_path = os.path.join(OUT_DIR, f"{TICKER}_H{H}_targetClose.npz")

d = np.load(npz_path, allow_pickle=True)

X_train = d["X_train"]
y_train = d["y_train"]
X_val   = d["X_val"]
y_val   = d["y_val"]
X_test  = d["X_test"]
y_test  = d["y_test"]

train_dates = d["train_target_date"]
val_dates   = d["val_target_date"]
test_dates  = d["test_target_date"]

print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_val  :", X_val.shape,   "y_val  :", y_val.shape)
print("X_test :", X_test.shape,  "y_test :", y_test.shape)
print("Train target date range:", train_dates[0], "->", train_dates[-1])
print("Test  target date range:", test_dates[0],  "->", test_dates[-1])

ModuleNotFoundError: No module named 'tensorflow'

In [ ]:
# Scale target y supaya training stabil

y_scaler = StandardScaler()
y_train_s = y_scaler.fit_transform(y_train.reshape(-1,1)).reshape(-1)
y_val_s   = y_scaler.transform(y_val.reshape(-1,1)).reshape(-1)
y_test_s  = y_scaler.transform(y_test.reshape(-1,1)).reshape(-1)

print("y_train scaled mean ~", y_train_s.mean(), "std ~", y_train_s.std())

In [ ]:
# Build model Attention-LSTM (temporal attention sederhana)

def build_attention_lstm(input_shape, lstm_units=64, dropout=0.2, lr=1e-3):
    inp = layers.Input(shape=input_shape)  # (lookback, n_features)

    x = layers.LSTM(lstm_units, return_sequences=True)(inp)
    x = layers.Dropout(dropout)(x)

    # ---- Temporal Attention ----
    # score: (batch, timesteps, 1)
    score = layers.Dense(1)(x)
    weights = layers.Softmax(axis=1, name="attn_weights")(score)  # softmax over time

    # context: sum_t (weights_t * x_t) => (batch, lstm_units)
    context = layers.Multiply()([x, weights])
    context = layers.Lambda(lambda t: tf.reduce_sum(t, axis=1), name="context")(context)

    # output (regression)
    out = layers.Dense(32, activation="relu")(context)
    out = layers.Dense(1, name="y_hat")(out)

    model = Model(inputs=inp, outputs=out)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss="mse",
        metrics=[tf.keras.metrics.MeanAbsoluteError(name="mae")]
    )
    return model

tf.keras.utils.set_random_seed(42)

model = build_attention_lstm(
    input_shape=X_train.shape[1:],  # (60, 1)
    lstm_units=64,
    dropout=0.2,
    lr=1e-3
)

model.summary()

In [ ]:
# Train

callbacks = [
    EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-6)
]

history = model.fit(
    X_train, y_train_s,
    validation_data=(X_val, y_val_s),
    epochs=100,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)

# Plot Training Curve
plt.figure()
plt.plot(history.history["loss"], label="train_loss")
plt.plot(history.history["val_loss"], label="val_loss")
plt.legend()
plt.title(f"Loss curve - {TICKER} H={H}")
plt.show()

In [ ]:
# Evaluate (inverse ke harga asli) + metrik MAE/RMSE/sMAPE

def smape(y_true, y_pred, eps=1e-8):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = (np.abs(y_true) + np.abs(y_pred) + eps)
    return 100.0 * np.mean(2.0 * np.abs(y_pred - y_true) / denom)

def rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.sqrt(np.mean((y_pred - y_true)**2)))

# pred (scaled)
yhat_train_s = model.predict(X_train).reshape(-1)
yhat_val_s   = model.predict(X_val).reshape(-1)
yhat_test_s  = model.predict(X_test).reshape(-1)

# inverse ke harga asli
yhat_train = y_scaler.inverse_transform(yhat_train_s.reshape(-1,1)).reshape(-1)
yhat_val   = y_scaler.inverse_transform(yhat_val_s.reshape(-1,1)).reshape(-1)
yhat_test  = y_scaler.inverse_transform(yhat_test_s.reshape(-1,1)).reshape(-1)

# metrik di skala asli
metrics = {
    "train_MAE": float(np.mean(np.abs(y_train - yhat_train))),
    "train_RMSE": rmse(y_train, yhat_train),
    "train_sMAPE": smape(y_train, yhat_train),

    "val_MAE": float(np.mean(np.abs(y_val - yhat_val))),
    "val_RMSE": rmse(y_val, yhat_val),
    "val_sMAPE": smape(y_val, yhat_val),

    "test_MAE": float(np.mean(np.abs(y_test - yhat_test))),
    "test_RMSE": rmse(y_test, yhat_test),
    "test_sMAPE": smape(y_test, yhat_test),
}

metrics

In [ ]:
# Plot prediksi vs aktual di test set

plt.figure(figsize=(12,4))
plt.plot(test_dates, y_test, label="Actual")
plt.plot(test_dates, yhat_test, label="Predicted")
plt.xticks(rotation=45)
plt.title(f"{TICKER} Close Forecast H={H} (Test)")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Save model + hasil metrik ke folder project

MODEL_DIR = "../models/attention_lstm"
FIG_DIR = "../figures"
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

model_path = os.path.join(MODEL_DIR, f"{TICKER}_H{H}_L{LOOKBACK}_close.keras")
model.save(model_path)

# simpan metrik ke csv
import pandas as pd
metrics_path = os.path.join(MODEL_DIR, f"metrics_{TICKER}_H{H}_L{LOOKBACK}_close.csv")
pd.DataFrame([metrics]).to_csv(metrics_path, index=False)

model_path, metrics_path